# 03 - Poisson Goals Model

Fit per-team attack and defense strengths, visualize them, draw a scoreline heatmap for a fixture, and backtest both models.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from wcpredictor.config import default_config
from wcpredictor.data import load_matches, load_teams, load_groups
from wcpredictor.data.preprocess import build_training_matches, team_match_counts

config = default_config()
plt.rcParams["figure.figsize"] = (9, 4.5)

import numpy as np
from wcpredictor.models import EloModel, PoissonModel
from wcpredictor.evaluation import backtest

In [ ]:
tr = build_training_matches(load_matches(config), config)
teams = load_teams(config)
poisson = PoissonModel(config).fit(tr)
strengths = poisson.strengths_table(teams)
print('home advantage factor:', round(np.exp(poisson.home_advantage), 3))
strengths.head(15)

## Attack vs defense

In [ ]:
fig, ax = plt.subplots()
ax.scatter(strengths['attack'], strengths['defense'], alpha=0.6)
for _, r in strengths.head(8).iterrows():
    ax.annotate(r['team'], (r['attack'], r['defense']))
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('attack strength'); ax.set_ylabel('defense strength')
ax.set_title('Team attack vs defense'); plt.tight_layout(); plt.show()

## Scoreline probabilities for a fixture

In [ ]:
home, away = 'France', 'Morocco'
matrix = poisson.scoreline_matrix(home, away, neutral=True)[:6, :6]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matrix, cmap='Blues', origin='lower')
ax.set_xlabel(f'{away} goals'); ax.set_ylabel(f'{home} goals')
ax.set_title(f'P(scoreline): {home} vs {away}')
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()
print('most likely score:', poisson.most_likely_score(home, away, True))

## Backtest: Elo vs Poisson

Walk-forward evaluation. Lower log-loss / RPS is better.

In [ ]:
elo_res = backtest(lambda: EloModel(config), tr, config, min_train=300)
poi_res = backtest(lambda: PoissonModel(config), tr, config, min_train=300)
pd.DataFrame([dict(model='Elo', **elo_res.as_dict()),
              dict(model='Poisson', **poi_res.as_dict())])